# Set microscope limits

This notebook writes the machine safety limits used by the Leica
navigator_expert driver. Limits are owned by navigator_expert.limits; no
controller or legacy motion.stage_config API is involved.

Edit the LIMITS table below, derive X/Y from four temporary LAS X
Point markers, then use the final cell to publish limits.json into a
date-stamped machine ProgramData snapshot. Save this notebook before the
final cell archives it under a timestamped filename.

How to read the table:

- A range is an inclusive [minimum, maximum].
- An allowed list contains the exact permitted values.
- An empty list means reviewed and intentionally unrestricted.
- Every listed key is required; missing and unknown keys are rejected.
- The independent hardcoded physical stage backstop always applies.


In [ ]:
import _bootstrap
from navigator_expert.limits.config import adopt_limits

LIMITS = {
    "x_um": {"range": [1000, 130000]},
    "y_um": {"range": [1000, 100000]},
    "z_galvo_um": {"range": [-250, 250]},
    "z_wide_um": {"range": [0, 8000]},
    "objective_slot": [],
    "set_zoom": [],
    "set_scan_speed": [],
    "set_scan_resonant": [],
    "set_scan_mode": [],
    "set_sequential_mode": [],
    "set_scan_field_rotation": [],
    "set_image_format": [],
    "set_z_stack_definition": [],
    "set_z_stack_step_size": [],
    "set_z_stack_size": [],
    "set_frame_accumulation": [],
    "set_frame_average": [],
    "set_line_accumulation": [],
    "set_line_average": [],
    "set_pinhole_airy": [],
    "set_detector_gain": [],
    "set_laser_intensity": [],
    "set_laser_shutter": [],
    "set_filter_wheel_slot": [],
    "set_filter_wheel_spectrum": [],
}

## Record X/Y limits from four LAS X points

In the active PythonInspect template in LAS X Navigator Expert, place exactly
four temporary Point markers at the safe sample boundaries. Do not leave scan
fields, focus markers, or other geometry in that template.

The cell below uses the Leica driver directly. It requires exactly four clean
Point markers, updates LIMITS x_um and y_um to their inclusive bounding
rectangle, preserves the saved LAS X experiment files, removes the temporary
markers only after a successful read, and prints the recorded X/Y ranges.


In [ ]:
import navigator_expert as drv
from navigator_expert.limits.adaptive import capture_adaptive_xy_limits

client = drv.connect_microscope(load_calibration=False)
captured_xy = capture_adaptive_xy_limits(client, remove_markers=True)
LIMITS.update(captured_xy["limits"])
print("Updated x_um:", LIMITS["x_um"])
print("Updated y_um:", LIMITS["y_um"])
print("Temporary Point markers removed:", captured_xy["markers_removed"])

## Save and publish

First save this notebook with Ctrl+S so the captured values are on disk.
Then run the final cell. It publishes limits.json, archives the timestamped
notebook under data/notebook, and archives the saved LAS X .xml, .rgn, and
.lrp files under data/template.


In [ ]:
published = adopt_limits(
    LIMITS,
    notebook_paths=[_bootstrap.NOTEBOOK_PATH],
    template_paths=captured_xy["template_paths"],
)
print("Published:", published["limits_path"])
print("Archived notebook:", published["notebook_paths"][0])
print("Archived template:", published["template_paths"])
published